# 04 Split Strategy — G1020, REFUGE, PAPILA (+ combined)

Creates leakage-aware train/val/test splits for the remaining manifests, following
`02_split_strategy.ipynb`: a group-based split on `split_group_id` (70/15/15,
seed 42), validated so no group crosses a split boundary. Per source:

- **G1020** — group split on file id (one image per group), like ORIGA.
- **PAPILA** — group split on **patient** (`split_group_id` is `PAPILA_RETxxx`, so
  both eyes of a patient always land in the same split — no leakage).
- **REFUGE** — honors its official `train`/`val`/`test` split rather than re-splitting
  (the canonical, comparable choice). Flip `REFUGE_MODE` to `"group_random"` to
  re-split it the same way as the others.

Finally, all sources (including the existing ORIGA split) are concatenated into one
`combined_segmentation_manifest_with_splits.csv` for training.

In [3]:
from pathlib import Path
import sys, os
import pandas as pd
from sklearn.model_selection import train_test_split

def find_project_root(start_path: Path | None = None) -> Path:
    start_path = start_path or Path.cwd()
    for path in [start_path, *start_path.parents]:
        if (path / "configs").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root containing configs/ and src/")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
RANDOM_SEED = 42
MANIFEST_DIR = PROJECT_ROOT / "data" / "processed" / "manifests"
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REFUGE_MODE = "official"   # "official" | "group_random"
CORE = ["dataset_key", "source_dataset", "file_id", "image_path", "mask_path",
        "split_group_id", "split"]

## Shared split + validation (same logic as `02`)

In [4]:
def group_split(manifest, seed=RANDOM_SEED):
    groups = (manifest[["split_group_id"]].drop_duplicates()
              .sort_values("split_group_id").reset_index(drop=True))
    train_g, temp_g = train_test_split(groups, test_size=0.30, random_state=seed, shuffle=True)
    val_g, test_g = train_test_split(temp_g, test_size=0.50, random_state=seed, shuffle=True)
    assign = pd.concat([train_g.assign(split="train"), val_g.assign(split="val"),
                        test_g.assign(split="test")], ignore_index=True)
    return manifest.merge(assign, on="split_group_id", how="left"), assign

def validate_splits(manifest_with_splits, expected_total):
    crossing = (manifest_with_splits.groupby("split_group_id")["split"].nunique() > 1).sum()
    checks = pd.DataFrame([
        {"check": "manifest_row_count_preserved",
         "passed": len(manifest_with_splits) == expected_total,
         "value": len(manifest_with_splits), "expected": expected_total},
        {"check": "no_missing_split_assignments",
         "passed": manifest_with_splits["split"].isna().sum() == 0,
         "value": int(manifest_with_splits["split"].isna().sum()), "expected": 0},
        {"check": "no_split_group_crosses_splits",
         "passed": crossing == 0, "value": int(crossing), "expected": 0},
    ])
    return checks

def process_source(name, manifest_file, mode="group_random"):
    manifest = pd.read_csv(MANIFEST_DIR / manifest_file, dtype={"file_id": str, "split_group_id": str})
    if mode == "official":
        manifest_with_splits = manifest.assign(split=manifest["refuge_official_split"])
        assign = (manifest_with_splits[["split_group_id", "split"]]
                  .drop_duplicates().sort_values("split_group_id").reset_index(drop=True))
    else:
        manifest_with_splits, assign = group_split(manifest)
    assign.to_csv(SPLIT_DIR / f"{name}_split_assignments.csv", index=False)
    manifest_with_splits.to_csv(MANIFEST_DIR / f"{name}_segmentation_manifest_with_splits.csv", index=False)
    checks = validate_splits(manifest_with_splits, len(manifest))
    checks.to_csv(SPLIT_DIR / f"{name}_split_validation_summary.csv", index=False)
    summary = manifest_with_splits.groupby("split").size().reindex(["train", "val", "test"]).fillna(0).astype(int)
    print(f"\n=== {name} ({mode}) ===")
    print("split sizes:", summary.to_dict())
    print("all checks passed:", bool(checks["passed"].all()))
    return manifest_with_splits

## Run each source

In [5]:
g1020_ws = process_source("g1020", "g1020_segmentation_manifest.csv", mode="group_random")
papila_ws = process_source("papila", "papila_segmentation_manifest.csv", mode="group_random")
refuge_ws = process_source("refuge", "refuge_segmentation_manifest.csv", mode=REFUGE_MODE)


=== g1020 (group_random) ===
split sizes: {'train': 714, 'val': 153, 'test': 153}
all checks passed: True

=== papila (group_random) ===
split sizes: {'train': 340, 'val': 74, 'test': 74}
all checks passed: True

=== refuge (official) ===
split sizes: {'train': 400, 'val': 400, 'test': 400}
all checks passed: True


## Combined manifest with splits (ORIGA + G1020 + REFUGE + PAPILA)

The existing ORIGA split is read back and concatenated with the three new ones to
a single training manifest. The combined train pool is the union of each source's
train split, preserving source balance and per-source leakage safety.

In [6]:
origa_ws = pd.read_csv(MANIFEST_DIR / "origa_segmentation_manifest_with_splits.csv",
                       dtype={"file_id": str, "split_group_id": str})
combined = pd.concat([df[CORE] for df in [origa_ws, g1020_ws, refuge_ws, papila_ws]],
                     ignore_index=True)
combined_path = MANIFEST_DIR / "combined_segmentation_manifest_with_splits.csv"
combined.to_csv(combined_path, index=False)

print("Saved:", combined_path)
print("\ncombined split x source (rows):")
print(combined.groupby(["source_dataset", "split"]).size().unstack(fill_value=0).reindex(columns=["train", "val", "test"]))
print("\ntotals by split:", combined.groupby("split").size().to_dict())

Saved: /Users/tylerhobbs/Documents/Virginia/Glaucoma Project/data/processed/manifests/combined_segmentation_manifest_with_splits.csv

combined split x source (rows):
split           train  val  test
source_dataset                  
G1020             714  153   153
ORIGA             455   97    98
PAPILA            340   74    74
REFUGE            400  400   400

totals by split: {'test': 725, 'train': 1909, 'val': 724}


In [7]:
# --- Pre-flight: confirm every path in the combined manifest resolves ---
_mf = PROJECT_ROOT / "data/processed/manifests/combined_segmentation_manifest_with_splits.csv"
_df = pd.read_csv(_mf)
_paths = pd.concat([_df["image_path"], _df["mask_path"]])
_missing = sorted({p for p in _paths if not (PROJECT_ROOT / p).exists()})
print(f"rows: {len(_df)} | unique files: {_paths.nunique()} | missing: {len(_missing)}")
for m in _missing[:10]:
    print("  MISSING:", m)
print(_df.groupby(["source_dataset", "split"]).size().unstack(fill_value=0))

rows: 3358 | unique files: 6716 | missing: 0
split           test  train  val
source_dataset                  
G1020            153    714  153
ORIGA             98    455   97
PAPILA            74    340   74
REFUGE           400    400  400


## Done

Per-source split assignments, validation summaries, and manifests-with-splits are
written alongside the ORIGA outputs, plus `combined_segmentation_manifest_with_splits.csv`.
The training notebooks read the combined manifest, filter by `split` (and optionally
`source_dataset`), and build datasets from `image_path` / `mask_path` — no folder
scanning or ad-hoc splitting.